# AllocationManager: Choosing the Right Strategy

The `AllocationManager` distributes available funds across envelopes using different allocation strategies. This notebook compares sorted (priority-based) and proportional allocation, and shows how to customize them for your needs.

**Topics covered:**
- Comparing sorted vs. proportional allocation
- Custom sort keys and weighting functions
- Understanding allocation results and metadata


In [1]:
from datetime import date
from decimal import Decimal
from sinkingfund.managers import AllocationManager
from sinkingfund.models import BillInstance, Envelope


## Setting Up Envelopes

Let's create some envelopes to compare allocation strategies:


In [2]:
# Create bill instances.
instances = [
    BillInstance(bill_id="urgent", service="Urgent Bill", due_date=date(2025, 2, 1), amount_due=Decimal("500.00")),
    BillInstance(bill_id="medium", service="Medium Priority", due_date=date(2025, 6, 1), amount_due=Decimal("800.00")),
    BillInstance(bill_id="later", service="Later Bill", due_date=date(2025, 10, 1), amount_due=Decimal("1200.00")),
]

# Create envelopes.
envelopes = [
    Envelope(bill_instance=inst, initial_allocation=Decimal("0.00")) 
    for inst in instances
]

print("Envelopes to allocate:")
for env in envelopes:
    print(f"  {env.bill_instance.service}: ${env.bill_instance.amount_due} due {env.bill_instance.due_date}")


Envelopes to allocate:
  Urgent Bill: $500.00 due 2025-02-01
  Medium Priority: $800.00 due 2025-06-01
  Later Bill: $1200.00 due 2025-10-01


## Sorted (Priority-Based) Allocation

Sorted allocation distributes funds in priority order. By default, it allocates to earliest due dates first (cascade strategy), but you can use custom sort keys.


In [3]:
# Create a copy of envelopes for sorted allocation.
sorted_envelopes = [
    Envelope(bill_instance=inst, initial_allocation=Decimal("0.00")) 
    for inst in instances
]

# Use sorted allocation (cascade = earliest due date first).
sorted_manager = AllocationManager()
sorted_manager.set_allocator(strategy="sorted", sort_key="cascade")
available_balance = Decimal("1000.00")

# Allocate returns an AllocationResult with allocations dictionary.
result = sorted_manager.allocate(envelopes=sorted_envelopes, balance=available_balance)

# Apply allocations to envelopes.
for envelope, allocation in result.envelopes.items():
    envelope.initial_allocation = allocation

print("=== Sorted Allocation (Earliest Due Date First) ===")
print(f"Available balance: ${available_balance}\n")
for envelope in sorted_envelopes:
    print(f"{envelope.bill_instance.service}: ${envelope.initial_allocation} allocated")
    print(f"  Remaining needed: ${envelope.bill_instance.amount_due - envelope.initial_allocation}")


=== Sorted Allocation (Earliest Due Date First) ===
Available balance: $1000.00

Urgent Bill: $500.00 allocated
  Remaining needed: $0.00
Medium Priority: $500.00 allocated
  Remaining needed: $300.00
Later Bill: $0.00 allocated
  Remaining needed: $1200.00


## Proportional Allocation

Proportional allocation distributes funds proportionally based on funding needs or bill amounts. Each envelope receives a share proportional to its target amount.


In [4]:
# Create a fresh copy of envelopes for proportional allocation.
proportional_envelopes = [
    Envelope(bill_instance=inst, initial_allocation=Decimal("0.00")) 
    for inst in instances
]

# Use proportional allocation.
# The proportional strategy requires a 'method' parameter.
# Options: "proportional" (by bill amount), "equal", "urgency", or "zero".
proportional_manager = AllocationManager()
proportional_manager.set_allocator(strategy="proportional", method="proportional")
result = proportional_manager.allocate(envelopes=proportional_envelopes, balance=available_balance)

# Apply allocations to envelopes.
for envelope, allocation in result.envelopes.items():
    envelope.initial_allocation = allocation

print("=== Proportional Allocation ===")
print(f"Available balance: ${available_balance}\n")

# Calculate proportions.
total_needed = sum(env.bill_instance.amount_due for env in proportional_envelopes)
for envelope in proportional_envelopes:
    proportion = (envelope.bill_instance.amount_due / total_needed) * 100
    print(f"{envelope.bill_instance.service}: ${envelope.initial_allocation} allocated ({proportion:.1f}% of target)")


=== Proportional Allocation ===
Available balance: $1000.00

Urgent Bill: $200.0 allocated (20.0% of target)
Medium Priority: $320.0 allocated (32.0% of target)
Later Bill: $480.0 allocated (48.0% of target)


## Custom Sort Keys

With sorted allocation, you can use custom sort keys to prioritize differently. Built-in options include "cascade" (due date) and "debt_snowball" (bill amount), or you can pass a custom function.


In [5]:
# Sort by amount (smallest first - "debt snowball" approach).
amount_sorted_envelopes = [
    Envelope(bill_instance=inst, initial_allocation=Decimal("0.00")) 
    for inst in instances
]

amount_manager = AllocationManager()
amount_manager.set_allocator(strategy="sorted", sort_key="debt_snowball", reverse=False)
result = amount_manager.allocate(envelopes=amount_sorted_envelopes, balance=available_balance)

In [6]:
# Apply allocations.
for envelope, allocation in result.envelopes.items():
    envelope.initial_allocation = allocation

print("=== Sorted by Amount (Smallest First) ===")
print(f"Available balance: ${available_balance}\n")
for envelope in amount_sorted_envelopes:
    print(f"{envelope.bill_instance.service} (${envelope.bill_instance.amount_due}): ${envelope.initial_allocation} allocated")

=== Sorted by Amount (Smallest First) ===
Available balance: $1000.00

Urgent Bill ($500.00): $500.00 allocated
Medium Priority ($800.00): $500.00 allocated
Later Bill ($1200.00): $0.00 allocated


In [7]:
# Sort by amount (largest first).
largest_first_envelopes = [
    Envelope(bill_instance=inst, initial_allocation=Decimal("0.00")) 
    for inst in instances
]

largest_manager = AllocationManager()
largest_manager.set_allocator(strategy="sorted", sort_key="debt_snowball", reverse=True)
result = largest_manager.allocate(envelopes=largest_first_envelopes, balance=available_balance)

# Apply allocations.
for envelope, allocation in result.envelopes.items():
    envelope.initial_allocation = allocation

print("\n=== Sorted by Amount (Largest First) ===")
for envelope in largest_first_envelopes:
    print(f"{envelope.bill_instance.service} (${envelope.bill_instance.amount_due}): ${envelope.initial_allocation} allocated")



=== Sorted by Amount (Largest First) ===
Urgent Bill ($500.00): $0.00 allocated
Medium Priority ($800.00): $0.00 allocated
Later Bill ($1200.00): $1000.00 allocated


## Understanding Allocation Results

The `allocate()` method returns an `AllocationResult` object containing the allocations and metadata about the strategy used.


In [8]:
# Get allocation result and examine metadata.
test_envelopes = [
    Envelope(bill_instance=inst, initial_allocation=Decimal("0.00")) 
    for inst in instances
]

test_manager = AllocationManager()
test_manager.set_allocator(strategy="sorted", sort_key="cascade")
result = test_manager.allocate(envelopes=test_envelopes, balance=available_balance)

# Apply allocations.
for envelope, allocation in result.envelopes.items():
    envelope.initial_allocation = allocation

print("=== Allocation Result ===")
print(f"Strategy: {result.metadata.get('strategy', 'unknown')}")
print(f"Sort key: {result.metadata.get('sort_key', 'unknown')}")
print(f"\nAllocations:")
for envelope in test_envelopes:
    print(f"  {envelope.bill_instance.service}: ${envelope.initial_allocation} allocated")


=== Allocation Result ===
Strategy: SortedAllocator
Sort key: <lambda>

Allocations:
  Urgent Bill: $500.00 allocated
  Medium Priority: $500.00 allocated
  Later Bill: $0.00 allocated


## Summary

**Key Takeaways:**

1. **Sorted Allocation**: Priority-based distribution
   - Use `sort_key="cascade"` for due date priority (earliest first)
   - Use `sort_key="debt_snowball"` for amount-based sorting (smallest first)
   - Use `reverse=True` to reverse sort order

2. **Proportional Allocation**: Mathematical distribution proportional to bill amounts
   - Requires `method` parameter: `method="proportional"` (by bill amount), `"equal"`, `"urgency"`, or `"zero"`
   - Each envelope receives a share based on its target amount
   - Ensures all envelopes get funding (proportional to need)

3. **Applying Allocations**: Allocations are returned in an `AllocationResult` object - apply them to envelope `initial_allocation` attributes as needed

4. **Setup**: Use `set_allocator()` to configure the strategy after creating the manager

**Next Steps:**
- See allocation in action within the complete SinkingFund workflow
- Explore advanced allocation customization options
